# Module 1.4 - Building Stateless and Stateful Chatbots

**Reference:** [`04-building-stateless-and-stateful-chatbots.md`](./04-building-stateless-and-stateful-chatbots.md)

## What you'll do in this notebook

- Build a **stateless** FAQ chatbot for a fictional cafe, DevBrew.
- Extend it into a **stateful** chatbot that remembers the conversation so far.
- Add a token budget that automatically prunes old turns when the history gets too long.
- Start your capstone project: a stateful chatbot for a domain of your choice.

## Setup

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import tiktoken

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set."

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
encoding = tiktoken.encoding_for_model("gpt-4o-mini")
print(f"OK: client ready, model = {MODEL}")

## Exercise 1 - Stateless FAQ bot

**Scenario:** DevBrew is a cafe serving coffee for programmers.

- Open 6 AM to 10 PM daily
- Located at 123 Code Street
- Menu highlights: Espresso, Cold Brew, and the "Debug Latte"

Write a system prompt that:
- Tells the bot its role (DevBrew FAQ assistant)
- Gives it the known facts
- Sets a friendly but concise personality
- Says how to handle off-topic questions

Then implement `StatelessChatbot`.

In [ ]:
SYSTEM_PROMPT = """
# TODO: write your DevBrew system prompt here.
"""

class StatelessChatbot:
    def __init__(self, system_prompt: str):
        self.system_prompt = system_prompt

    def chat(self, user_message: str) -> str:
        # TODO: call the API with [system_prompt, user_message] and return the reply.
        raise NotImplementedError

bot = StatelessChatbot(SYSTEM_PROMPT)
print("Hours:", bot.chat("What are your hours?"))
print("Menu: ", bot.chat("Do you have cold brew?"))
print("Off-topic:", bot.chat("What is the meaning of life?"))  # should politely redirect

**Try it:** ask the stateless bot this sequence:

1. "Tell me about the Debug Latte."
2. "How much does it cost?"

The second question refers to "it" — because the bot is stateless, it has no way to know what "it" is. Observe the failure mode. Exercise 2 fixes this.

## Exercise 2 - Stateful chatbot

Extend the design so the bot keeps a running `conversation_history` list and sends the whole list with every call.

In [ ]:
class StatefulChatbot:
    def __init__(self, system_prompt: str):
        self.system_prompt = system_prompt
        # TODO: initialise self.conversation_history with the system message.
        self.conversation_history: list[dict] = []

    def chat(self, user_message: str) -> str:
        # TODO:
        # 1. append the user message to conversation_history
        # 2. call the API with the full history
        # 3. append the assistant reply to conversation_history
        # 4. return the reply
        raise NotImplementedError

    def get_history(self) -> list[dict]:
        # TODO: return history excluding the system message.
        raise NotImplementedError

    def reset(self) -> None:
        # TODO: clear history back to just the system message.
        raise NotImplementedError


bot = StatefulChatbot(SYSTEM_PROMPT)
print(bot.chat("Tell me about your cold brew."))
print(bot.chat("How much does that cost?"))  # "that" should resolve to cold brew
print(bot.chat("I'll take two."))             # should understand the order

assert len(bot.get_history()) == 6, "Expected 3 user + 3 assistant messages"
bot.reset()
assert len(bot.get_history()) == 0, "reset() should clear history"
print("\nOK: state and reset work")

## Exercise 3 - Token-aware history pruning

A stateful bot grows without bound. Add a token budget: when the history exceeds `max_history_tokens`, drop the oldest user+assistant pair (but **never** the system prompt).

In [ ]:
class TokenManagedChatbot(StatefulChatbot):
    def __init__(self, system_prompt: str, max_history_tokens: int = 1000):
        super().__init__(system_prompt)
        self.max_history_tokens = max_history_tokens
        self.encoding = tiktoken.encoding_for_model("gpt-4o-mini")

    def count_tokens(self, messages: list[dict]) -> int:
        # TODO: return the total token count across all messages' content fields.
        raise NotImplementedError

    def prune_history(self) -> None:
        # TODO: while count_tokens(self.conversation_history) > self.max_history_tokens,
        # remove the oldest non-system messages in user/assistant pairs.
        # Keep self.conversation_history[0] (the system message) intact.
        raise NotImplementedError

    def chat(self, user_message: str) -> str:
        self.conversation_history.append({"role": "user", "content": user_message})
        self.prune_history()

        response = client.chat.completions.create(
            model=MODEL,
            messages=self.conversation_history,
        )
        reply = response.choices[0].message.content
        self.conversation_history.append({"role": "assistant", "content": reply})
        return reply


bot = TokenManagedChatbot(SYSTEM_PROMPT, max_history_tokens=400)
for i in range(8):
    bot.chat(f"Tell me about coffee drink number {i}.")
    total = bot.count_tokens(bot.conversation_history)
    print(f"Turn {i+1:2}: {len(bot.get_history()):2} non-system messages, {total:4} tokens")

Watch the message count stop growing once the budget is hit. Now reflect: if the user said something important on turn 1, do you still have access to it on turn 8? This is the motivation for summarisation and selective memory in Module 4.

## Capstone checkpoint 1

By the end of this section you should have the foundation of your capstone chatbot. Copy the `TokenManagedChatbot` into your capstone project folder and adapt it:

- Replace the DevBrew system prompt with one for **your** chosen domain (customer support, HR onboarding, tutor, etc.).
- Define 3-5 core capabilities the bot should cover.
- Add the retry logic from Exercise 1.3.4.
- Have a test conversation of at least 10 turns that demonstrates the capabilities.

You'll add retrieval in Module 2, caching in Module 3, and better memory in Module 4.

## What's next

Module 2 tackles one of the hardest limits of an LLM: it only knows what it was trained on. In `module_2/01-the-rag-paradigm.ipynb` we introduce Retrieval-Augmented Generation — giving the chatbot the ability to look up information in your own documents.